# R11 · Multi-core query timing (complete queries, every dataset)

How each step of a complete query scales with the number of physical cores T, on every dataset: the whole-matrix
logarithm (one dense eigendecomposition, T BLAS threads), the query's block logarithms (independent per niche and block,
on a pool of T worker processes), the whole-matrix and Spindle-naive scans (rows split over the pool) and Spindle's
search, both on one core (as R4) and split by niche over the pool (`FlatSpindle.probe_niche` / `scan_niche`,
same output). Every parallel step is asserted equal, bit for bit, to its single-thread counterpart. At T = 1 the block
logs are also timed by the pool's per-block route on one core (the baseline of their scaling). Definitions:
`experiments/common/parallel_timing.py`. The paper's single-thread benchmark stays R4; T = 1 here repeats its protocol.

* **Prerequisites:** `R0_index_cache`, `R4_timing` (for the T = 1 check).
* **Mode:** environment variable `SPINDLE_PARALLEL` = `pilot` (brain only, seed 0, 20 queries, T = 1, 8, 64; always
  rewritten) or `full` (every dataset, seeds 0–4, first 200 held-out queries, T = 1, 2, 4, 8, 16, 32, 64).
* **Writes:** `results/parallel/`. Per dataset: `per_dataset/<key>/timing.csv` (one row per T, seed, query: ms of every
  step). Pooled: `timing.csv`, `seeds.csv` (median per dataset, T, seed) and `summary.csv` (mean and s.d. over seeds,
  end-to-end totals, scaling over T = 1, end-to-end speed-ups of Spindle). Existing per-dataset files are skipped.
* **Run:** `sbatch --export=ALL,SPINDLE_PARALLEL=full experiments/common/parallel_nb.sbatch experiments/compute/R11_parallel.ipynb`
  (65 physical cores of ROSALIND-CPU1: 64 workers + the main process, one hardware thread per core; workers busy-wait,
  see parallel_timing.py; the node is shared with other users' jobs). About 1–1.5 h; peak memory about 10 GB (brain).


In [1]:
import gc
import os
import subprocess
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets.yaml").exists())
sys.path.insert(0, str(ROOT / "experiments" / "common"))
import pandas as pd
from threadpoolctl import threadpool_info

import data
import parallel_timing as pt
import results_io as rio

TOPIC = "parallel"
MODE = os.environ.get("SPINDLE_PARALLEL", "pilot")
if MODE == "pilot":
    KEYS, SEEDS, N_QUERIES, THREADS, NAME, FORCE = ["brain_cancer"], [0], 20, [1, 8, 64], "pilot_timing.csv", True
else:
    KEYS, SEEDS, N_QUERIES, THREADS, NAME, FORCE = data.indexed_keys(), data.SEEDS, 200, [1, 2, 4, 8, 16, 32, 64], "timing.csv", False
print(data.stamp(), "| mode", MODE, "| job", os.environ.get("SLURM_JOB_ID"), "| cpus", len(os.sched_getaffinity(0)))
print([(i["internal_api"], i["version"], i["threading_layer"], i["num_threads"]) for i in threadpool_info()])
print(subprocess.run(["bash", "-c", "cat /proc/loadavg; squeue -h -w $(hostname -s) -o '%i %u %C %j' | head -40"],
                     capture_output=True, text=True).stdout)

{'commit': 'ac3d647'} | mode full | job 26713 | cpus 130
[('openblas', '0.3.34', 'pthreads', 64), ('openblas', '0.3.31.dev', 'pthreads', 64)]
1.20 1.11 1.04 8/4165 3424075
26713 ghoss18 130 spindle_timing



## Time every dataset (skipped where its file exists)

In [2]:
for key in KEYS:
    f = rio.per_dataset(TOPIC, key) / NAME
    if f.exists() and not FORCE:
        print(f"{key}: exists, skipped")
        continue
    df = pt.dataset_parallel(key, THREADS, N_QUERIES, SEEDS, log=lambda m: print(m, flush=True))
    df.to_csv(f, index=False)
    rio.register(TOPIC, f, f"{key}: one row per (threads, seed, query), ms of every step on T physical cores ({MODE})",
                 len(df))
    del df
    gc.collect()

64 physical cores for workers [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 60, 61, 62, 63], main on 64; timing T = [1, 2, 4, 8, 16, 32, 64]
skin_melanoma seed 0 T 1: 93 queries; median ms: logs whole 9.51, block 4.67; scans whole 16.55, naive 1.00; spindle 0.229 (1 core), 0.229 (niches on T cores)
seed 0 T 2: block-log balance 1.00, Spindle niche balance 1.06 (max / mean load), 4 Spindle-naive chunks
skin_melanoma seed 0 T 2: 93 queries; median ms: logs whole 10.03, block 2.01; scans whole 8.33, naive 0.57; spindle 0.162 (1 core), 0.194 (niches on T cores)
seed 0 T 4: block-log balance 1.02, Spindle niche balance 1.47 (max / mean load), 5 Spindle-naive chunks
skin_melanoma seed 0 T 4: 93 queries; median ms: logs whole 9.32, block 1.11; scans whole 5.36, naive 0.33; spindle 0.164 (1 core), 0.176 (niches 

## Pooled tables

In [3]:
tm = pd.concat([pd.read_csv(rio.topic(TOPIC) / "per_dataset" / k / NAME) for k in KEYS], ignore_index=True)
seeds, summ = pt.summary(tm)
pre = "pilot_" if MODE == "pilot" else ""
rio.write_csv(tm, TOPIC, f"{pre}timing.csv", "one row per (dataset, threads, seed, query): ms of every step on T cores")
rio.write_csv(seeds, TOPIC, f"{pre}seeds.csv", "median ms of every step and end-to-end total per (dataset, threads, seed)")
rio.write_csv(summ, TOPIC, f"{pre}summary.csv", "mean and s.d. over seeds of the per-seed medians, end-to-end totals, "
              "scaling over T = 1, end-to-end speed-ups of Spindle");

wrote results/parallel/timing.csv: 46935 rows
wrote results/parallel/seeds.csv: 280 rows
wrote results/parallel/summary.csv: 56 rows


## Summary and the T = 1 check against R4

In [4]:
pd.set_option("display.width", 250)
cols = ["key", "threads", "e2e_whole_ms", "e2e_naive_ms", "e2e_spindle_ms", "speedup_e2e_spindle_vs_whole",
        "speedup_e2e_spindle_vs_naive", "qlog_whole_ms", "qlog_block_ms", "spindle_par_ms"]
display(summ[cols].round(2))
r4 = pd.read_csv(ROOT / "results" / "timing" / "whole_tile.csv")
r4 = r4[r4.key.isin(KEYS) & r4.seed.isin(SEEDS) & (r4["query"] < N_QUERIES)]
steps = [c for c in pt.STEPS if c in r4.columns]
r4m = r4.groupby(["key", "seed"])[steps].median().groupby("key").mean()
one = seeds[seeds.threads == 1].groupby("key")[steps].mean()
display((one / r4m).round(3).rename(columns=lambda c: c + " (R11 T=1 / R4)"))

,key,threads,e2e_whole_ms,e2e_naive_ms,e2e_spindle_ms,speedup_e2e_spindle_vs_whole,speedup_e2e_spindle_vs_naive,qlog_whole_ms,qlog_block_ms,spindle_par_ms
0,brain_cancer,1,458.15,40.73,28.24,16.22,1.44,14.23,25.26,3.00
1,brain_cancer,2,237.52,17.11,10.76,22.08,1.59,12.99,8.79,2.00
2,brain_cancer,4,126.60,9.56,6.05,20.94,1.58,11.72,4.60,1.45
3,brain_cancer,8,93.96,6.91,3.68,25.50,1.87,10.85,2.50,1.18
4,brain_cancer,16,72.35,5.14,2.51,28.81,2.05,11.22,1.51,1.01
5,brain_cancer,32,44.38,3.36,2.02,21.92,1.66,12.05,0.96,1.05
6,brain_cancer,64,101.76,2.45,1.95,52.15,1.25,81.45,0.76,1.19
7,breast_cancer,1,29.31,5.30,4.01,7.31,1.32,5.32,3.61,0.40
8,breast_cancer,2,17.79,2.61,1.93,9.22,1.35,5.53,1.64,0.29
9,breast_cancer,4,13.69,1.56,1.10,12.44,1.42,5.77,0.86,0.23


,qlog_whole_ms (R11 T=1 / R4),qlog_block_ms (R11 T=1 / R4),whole_ms (R11 T=1 / R4),naive_ms (R11 T=1 / R4),spindle_ms (R11 T=1 / R4)
key,,,,,
brain_cancer,0.983,0.986,0.983,1.026,1.034
breast_cancer,0.981,0.967,0.997,0.993,1.021
kidney_nondiseased,0.971,0.958,1.011,1.012,0.964
lung_cancer,0.968,0.968,1.004,1.010,1.018
lymph_node,0.957,0.964,1.015,1.026,1.029
lymph_node_5k,0.978,0.983,1.020,1.029,1.035
pancreatic_cancer,0.968,0.958,0.997,1.021,1.016
skin_melanoma,0.982,0.961,0.990,0.973,0.971


In [5]:
rio.write_manifest(TOPIC, "R11_parallel.ipynb", inputs=["results/_cache (R0_index_cache)", "results/timing (R4_timing)"],
                   notes=f"every dataset on T physical cores of ROSALIND-CPU1 (one hardware thread per core; node shared "
                         f"with other users' jobs); mode {MODE}.")

wrote results/parallel/manifest.json: 11 files; peak memory 10.83 GB


{'topic': 'parallel',
 'notebook': 'R11_parallel.ipynb',
 'date': '2026-11-12 23:17',
 'commit': 'ac3d647',
 'inputs': ['results/_cache (R0_index_cache)', 'results/timing (R4_timing)'],
 'peak_rss_gb': 10.83,
 'notes': "every dataset on T physical cores of ROSALIND-CPU1 (one hardware thread per core; node shared with other users' jobs); mode full.",
 'files': {'per_dataset/brain_cancer/timing.csv': {'description': 'brain_cancer: one row per (threads, seed, query), ms of every step on T physical cores (full)',
   'rows': 7000,
   'bytes': 1371058},
  'per_dataset/breast_cancer/timing.csv': {'description': 'breast_cancer: one row per (threads, seed, query), ms of every step on T physical cores (full)',
   'rows': 7000,
   'bytes': 1379332},
  'per_dataset/kidney_nondiseased/timing.csv': {'description': 'kidney_nondiseased: one row per (threads, seed, query), ms of every step on T physical cores (full)',
   'rows': 2730,
   'bytes': 544339},
  'per_dataset/lung_cancer/timing.csv': {'descr